# Caderno 02: Staging, Pré-processamento e Data Lineage

**Objetivo:** Preservar a base bruta em um banco de dados relacional, criar a tabela analítica normalizada, isolar anomalias contábeis (estornos) e exportar os datasets limpos.

**Padrão Empírico ACM SIGSOFT (Data Science):** * *Explains how the data was pre-processed, filtered, and categorized* (Explica como os dados foram pré-processados, filtrados e categorizados).

**Decisão Arquitetural (Data Lineage):**
Para mitigar ameaças à validade relacionadas à manipulação indevida dos dados, implementou-se uma arquitetura de banco de dados SQLite em duas camadas com dupla exportação:
1. **Camada Staging (`stg_ceap`):** Ingestão do dado exatamente como fornecido pelo Senado, garantindo a imutabilidade do registro original.
2. **Camada Fato (`fato_despesa` e `estornos_ceap`):** Aplicação de engenharia de formatação (tipagem, limpeza de strings e adequação monetária), documentada no módulo `src/preprocess.py`. Valores negativos/zerados (estornos) foram isolados em uma tabela separada para não contaminar a análise estatística. O dado é salvo no SQLite para auditoria e em `.csv` para alimentar as etapas de Machine Learning.

In [1]:
import pandas as pd
import sqlite3
import glob
import sys
from pathlib import Path

# Adiciona a pasta src ao path do sistema para importar os módulos
ROOT_DIR = Path().resolve().parent
sys.path.append(str(ROOT_DIR))

# Importa as funções e constantes de limpeza
from src.preprocess import (
    preprocessar_dataframe, 
    COLUNAS_FATO, 
    COLUNAS_ESTORNO, 
    salvar_relatorio_preprocessamento
)

# Configuração de Caminhos
RAW_DIR = ROOT_DIR / 'data' / 'raw'
DB_PATH = ROOT_DIR / 'database' / 'ceap.db'
PROCESSED_DIR = ROOT_DIR / 'data' / 'processed'
LOGS_DIR = ROOT_DIR / 'logs'

# Garante que as pastas existem
RAW_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

print("Estabelecendo conexão com SQLite...")
conn = sqlite3.connect(DB_PATH)

Estabelecendo conexão com SQLite...


### 2.1 Carga RAW (Staging Area)
Todos os arquivos CSV da pasta `data/raw` são lidos como *strings* (texto puro). Adicionamos a coluna `source_file` para manter a rastreabilidade da linha até o arquivo original.

In [2]:
# Busca todos os arquivos coletados
arquivos_csv = glob.glob(str(RAW_DIR / 'ceap_*.csv'))
lista_dfs = []

print("Agora, iniciando a leitura dos CSVs brutos...")
for arquivo in arquivos_csv:
    # Parâmetros específicos para o CSV do SF
    df_ano = pd.read_csv(
        arquivo, 
        sep=';', 
        encoding='utf-8', 
        dtype=str, 
        on_bad_lines='skip'
    )
    # Rastreabilidade
    df_ano['source_file'] = Path(arquivo).name
    lista_dfs.append(df_ano)

# Empilha os anos e salva no banco de dados
df_raw_completo = pd.concat(lista_dfs, ignore_index=True)
df_raw_completo.to_sql('stg_ceap', conn, if_exists='replace', index=False)

print("Carga RAW concluída com sucesso!")
print(f"  -> Total de registros na tabela 'stg_ceap': {len(df_raw_completo):,}")

Agora, iniciando a leitura dos CSVs brutos...
Carga RAW concluída com sucesso!
  -> Total de registros na tabela 'stg_ceap': 88,180


### 2.2 Pré-processamento, Tipagem e Exportação
Aplicação do pipeline de sanitização (`preprocessar_dataframe`). As seguintes operações metodológicas são realizadas:
1. **Conversão Monetária:** Substituição do padrão brasileiro (`2.102,89`) para float computacional (`2102.89`).
2. **Sanitização de Documentos:** Remoção de máscaras de CNPJ/CPF via Expressão Regular.
3. **Datas:** Conversão da string (ISO) para o tipo `datetime` nativo do pandas.
4. **Padronização Categórica:** Transformação para caixa alta e remoção de espaços nas extremidades (*strip*) e múltiplos internos.
5. **Filtro de Estornos:** Separação de registros com valor <= 0 para não distorcer estatísticas.
6. **Relatório de Qualidade:** Geração de log com as métricas de conversão.

In [3]:
print("Aplicando regras de sanitização/limpeza")

# Chama a função principal de limpeza
df_fato_bruto, df_estornos_bruto, relatorio = preprocessar_dataframe(df_raw_completo)

# Filtra os DataFrames apenas com as colunas definidas para fato e estornos
df_fato = df_fato_bruto[COLUNAS_FATO].copy()
df_estornos = df_estornos_bruto[COLUNAS_ESTORNO].copy()

# 1. Persiste as tabelas no banco de dados SQLite
df_fato.to_sql('fato_despesa', conn, if_exists='replace', index=False)
df_estornos.to_sql('estornos_ceap', conn, if_exists='replace', index=False)

# 2. Exporta fisicamente para a pasta data/processed
caminho_csv_fato = PROCESSED_DIR / 'ceap_56_legislatura_fato.csv'
caminho_csv_estornos = PROCESSED_DIR / 'ceap_56_legislatura_estornos.csv'

df_fato.to_csv(caminho_csv_fato, index=False, encoding='utf-8', sep=';')
df_estornos.to_csv(caminho_csv_estornos, index=False, encoding='utf-8', sep=';')

# 3. Salva o Relatório de Qualidade de Dados (Sanity Check)
caminho_log = salvar_relatorio_preprocessamento(relatorio, LOGS_DIR)

print("Normalização e exportação concluídas com sucesso!")
print(f"  -> Total de registros na tabela 'fato_despesa': {len(df_fato):,}")
print(f"  -> Total de registros na tabela 'estornos_ceap': {len(df_estornos):,}")
print(f"  -> Relatório de qualidade salvo em: {caminho_log.name}")

# Fechando a conexão
conn.close()
print("\nPipeline de ingestão finalizado e banco de dados fechado.")

Aplicando regras de sanitização/limpeza
Normalização e exportação concluídas com sucesso!
  -> Total de registros na tabela 'fato_despesa': 88,111
  -> Total de registros na tabela 'estornos_ceap': 69
  -> Relatório de qualidade salvo em: preprocessamento_qualidade.json

Pipeline de ingestão finalizado e banco de dados fechado.
